In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# Path relative to the notebook location
clean_dir = "Drone Logs/Clean CSVs"
if not os.path.exists(clean_dir):
    if os.path.exists("Data/Drone Logs/Clean CSVs"):
        clean_dir = "Data/Drone Logs/Clean CSVs"
    else:
        print("Could not find Clean CSVs directory.")

# Data storage
flight_summaries = []
sample_flight_data = None
longest_duration = 0

print("Reading and summarizing flight files...")

if os.path.exists(clean_dir):
    for filename in os.listdir(clean_dir):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(clean_dir, filename)
                df = pd.read_csv(file_path, low_memory=False)
                
                # Metrics to extract
                metrics = {}
                
                # 1. Duration
                if 'OSD.flyTime' in df.columns:
                    duration = df['OSD.flyTime'].max() - df['OSD.flyTime'].min()
                    metrics['Duration (s)'] = duration
                    
                    # Keep the longest flight as a sample for detailed plotting
                    if duration > longest_duration:
                        longest_duration = duration
                        sample_flight_data = df.copy()
                        sample_flight_name = filename
                
                # 2. Altitude
                if 'OSD.height' in df.columns:
                    metrics['Max Height (m)'] = df['OSD.height'].max()
                elif 'OSD.altitude' in df.columns:
                     metrics['Max Height (m)'] = df['OSD.altitude'].max()

                # 3. Speed (Horizontal)
                if 'OSD.xSpeed' in df.columns and 'OSD.ySpeed' in df.columns:
                    # Calculate horizontal speed magnitude
                    df['horizontal_speed'] = np.sqrt(df['OSD.xSpeed']**2 + df['OSD.ySpeed']**2)
                    metrics['Max Horizontal Speed (m/s)'] = df['horizontal_speed'].max()
                    metrics['Avg Horizontal Speed (m/s)'] = df['horizontal_speed'].mean()
                
                # 4. Battery Consumption
                if 'BATTERY.chargeLevel' in df.columns:
                    start_batt = df['BATTERY.chargeLevel'].iloc[0]
                    end_batt = df['BATTERY.chargeLevel'].iloc[-1]
                    metrics['Battery Consumed (%)'] = start_batt - end_batt
                    metrics['Min Battery Level (%)'] = df['BATTERY.chargeLevel'].min()
                
                # 5. Satellites
                if 'OSD.gpsNum' in df.columns:
                    metrics['Avg Satellite Count'] = df['OSD.gpsNum'].mean()
                
                flight_summaries.append(metrics)
                
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    print(f"Processed {len(flight_summaries)} flights.")
    summary_df = pd.DataFrame(flight_summaries)
else:
    print("Directory not found.")

In [ ]:
if not summary_df.empty:
    # Set up a generic plotting style
    plt.style.use('default') # Reset to default just in case
    fig, axes = plt.subplots(3, 2, figsize=(15, 18))
    fig.suptitle(f'Flight Data Overview (N={len(summary_df)} Flights)', fontsize=16)

    # 1. Duration Distribution
    if 'Duration (s)' in summary_df.columns:
        axes[0, 0].hist(summary_df['Duration (s)'].dropna(), bins=30, color='skyblue', edgecolor='black')
        axes[0, 0].set_title('Flight Duration Distribution')
        axes[0, 0].set_xlabel('Duration (seconds)')
        axes[0, 0].set_ylabel('Count')

    # 2. Max Height Distribution
    if 'Max Height (m)' in summary_df.columns:
        axes[0, 1].hist(summary_df['Max Height (m)'].dropna(), bins=30, color='lightgreen', edgecolor='black')
        axes[0, 1].set_title('Max Flight Height Distribution')
        axes[0, 1].set_xlabel('Height (meters)')
        axes[0, 1].set_ylabel('Count')

    # 3. Max Speed Distribution
    if 'Max Horizontal Speed (m/s)' in summary_df.columns:
        axes[1, 0].hist(summary_df['Max Horizontal Speed (m/s)'].dropna(), bins=30, color='salmon', edgecolor='black')
        axes[1, 0].set_title('Max Horizontal Speed Distribution')
        axes[1, 0].set_xlabel('Speed (m/s)')
        axes[1, 0].set_ylabel('Count')

    # 4. Battery vs Duration
    if 'Duration (s)' in summary_df.columns and 'Battery Consumed (%)' in summary_df.columns:
        axes[1, 1].scatter(summary_df['Duration (s)'], summary_df['Battery Consumed (%)'], alpha=0.5, color='orange')
        axes[1, 1].set_title('Battery Consumption vs Duration')
        axes[1, 1].set_xlabel('Duration (s)')
        axes[1, 1].set_ylabel('Battery Consumed (%)')
        axes[1, 1].grid(True, alpha=0.3)

    # 5. Satellite Count
    if 'Avg Satellite Count' in summary_df.columns:
        axes[2, 0].hist(summary_df['Avg Satellite Count'].dropna(), bins=20, color='violet', edgecolor='black')
        axes[2, 0].set_title('Average Satellite Count Distribution')
        axes[2, 0].set_xlabel('Satellite Count')
        axes[2, 0].set_ylabel('Count')
    
    # Hide the 6th subplot if empty
    axes[2, 1].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    print("No summary data available to plot.")

In [ ]:
# Detailed plot for a single representative flight (The longest one)
if sample_flight_data is not None:
    print(f"Plotting detailed data for sample flight: {sample_flight_name} (Duration: {longest_duration:.1f}s)")
    
    # Ensure time column is sorted
    if 'OSD.flyTime' in sample_flight_data.columns:
        sample_flight_data = sample_flight_data.sort_values('OSD.flyTime')
        time = sample_flight_data['OSD.flyTime']
        
        fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
        
        # 1. Altitude
        if 'OSD.height' in sample_flight_data.columns:
            axes[0].plot(time, sample_flight_data['OSD.height'], color='green', label='Height')
            axes[0].set_ylabel('Height (m)')
            axes[0].set_title('Altitude Profile')
            axes[0].legend(loc='upper right')
            axes[0].grid(True)
            
        # 2. Speed
        if 'horizontal_speed' in sample_flight_data.columns:
            axes[1].plot(time, sample_flight_data['horizontal_speed'], color='red', label='Horizontal Speed')
            if 'OSD.zSpeed' in sample_flight_data.columns:
                 axes[1].plot(time, sample_flight_data['OSD.zSpeed'], color='blue', alpha=0.5, label='Vertical Speed')
            axes[1].set_ylabel('Speed (m/s)')
            axes[1].set_title('Speed Profile')
            axes[1].legend(loc='upper right')
            axes[1].grid(True)

        # 3. Battery
        if 'BATTERY.chargeLevel' in sample_flight_data.columns:
            axes[2].plot(time, sample_flight_data['BATTERY.chargeLevel'], color='orange', label='Charge Level')
            axes[2].set_ylabel('Battery (%)')
            axes[2].set_title('Battery Drain')
            axes[2].legend(loc='upper right')
            axes[2].grid(True)
            # Add voltage on secondary axis if available
            if 'BATTERY.voltage' in sample_flight_data.columns:
                ax2_twin = axes[2].twinx()
                ax2_twin.plot(time, sample_flight_data['BATTERY.voltage'], color='purple', linestyle='--', alpha=0.5, label='Voltage')
                ax2_twin.set_ylabel('Voltage (V)')
                # Handle legends for twin axes
                lines, labels = axes[2].get_legend_handles_labels()
                lines2, labels2 = ax2_twin.get_legend_handles_labels()
                axes[2].legend(lines + lines2, labels + labels2, loc='upper right')

        # 4. Signal / Satellites
        if 'RC.downlinkSignal' in sample_flight_data.columns:
            axes[3].plot(time, sample_flight_data['RC.downlinkSignal'], color='blue', label='RC Signal')
        if 'OSD.gpsNum' in sample_flight_data.columns:
            axes[3].plot(time, sample_flight_data['OSD.gpsNum'], color='grey', linestyle=':', label='Satellites')
        axes[3].set_ylabel('Signal / Count')
        axes[3].set_title('Signal & GPS Status')
        axes[3].set_xlabel('Time (s)')
        axes[3].legend(loc='upper right')
        axes[3].grid(True)
        
        plt.tight_layout()
        plt.show()
else:
    print("No sample flight data available.")
